In [ ]:

import PIL
import PIL.Image
import tensorflow as tf
import pathlib

import matplotlib.pyplot as plt
import numpy as np

In [ ]:
data_dir = pathlib.Path('Data/OT_Test/OT_images')
print(data_dir)

image_count = len(list(data_dir.glob('*/*.jpg')))
print(image_count)

In [ ]:
batch_size = 6
img_height = 244
img_width = 244

train_ds = tf.keras.preprocessing.image_dataset_from_directory(
  data_dir,
  validation_split=0.3,
  subset="training",
  seed=123,
  image_size=(img_height, img_width),
  batch_size=batch_size)

val_ds = tf.keras.preprocessing.image_dataset_from_directory(
  data_dir,
  validation_split=0.3,
  subset="validation",
  seed=123,
  image_size=(img_height, img_width),
  batch_size=batch_size)

In [ ]:
num_classes = 2

model = tf.keras.Sequential([
  tf.keras.layers.Rescaling(1./255),
  tf.keras.layers.Conv2D(16, 3, activation='relu'),
  tf.keras.layers.MaxPooling2D(),
  tf.keras.layers.Conv2D(16, 3, activation='relu'),
  tf.keras.layers.MaxPooling2D(),
  tf.keras.layers.Conv2D(16, 3, activation='relu'),
  tf.keras.layers.MaxPooling2D(),
  tf.keras.layers.Flatten(), 
  tf.keras.layers.Dense(128, activation='relu'),
  tf.keras.layers.Dense(num_classes)
])

In [ ]:
model.compile(
  optimizer='adam',
    loss=tf.losses.SparseCategoricalCrossentropy(from_logits=True),
  metrics=['accuracy'])

In [ ]:
model.fit(
  train_ds,
  validation_data=val_ds,
  epochs=4
)

In [ ]:
model.summary()

In [ ]:
model.evaluate(
  val_ds
)

In [ ]:
# img_path = "Data/F1_Test/F1_images/F1_Pass/F1_Horiz_00110.jpg"
img_path = "Data/OT_Test/OT_images/OT_Fail/OT_Fail1_00110.jpg"

img = tf.keras.utils.load_img(
    img_path,
    target_size=(img_height, img_width)
)

img_array = tf.keras.utils.img_to_array(img)
img_array = np.expand_dims(img_array, axis=0)

plt.imshow(img)
plt.axis("off")
plt.title("Input Image")
plt.show()

logits = model.predict(img_array)
probs = tf.nn.softmax(logits[0]).numpy()

pred_class = np.argmax(probs)
confidence = probs[pred_class]

print("Logits:", logits)
print("Probabilities:", probs)
print("Predicted class index:", pred_class)
print("Predicted label:", train_ds.class_names[pred_class])
print("Confidence:", confidence)

In [ ]:
model.save("Export/ot_model.keras")

In [ ]:
img_path = "Data/OT_Test/OT_images/OT_Pass/OT_Pass1_00010.jpg"
# img_path = "Data/F1_Test/F1_images/F1_Fail/F1_None_00110.jpg"

img = tf.keras.utils.load_img(
    img_path,
    target_size=(img_height, img_width)
)

img_array = tf.keras.utils.img_to_array(img)
img_array = np.expand_dims(img_array, axis=0)

plt.imshow(img)
plt.axis("off")
plt.title("Input Image")
plt.show()

loaded_model = tf.keras.models.load_model("Export/test_model.keras")
logits = loaded_model.predict(img_array)
probs = tf.nn.softmax(logits[0]).numpy()

pred_class = np.argmax(probs)
confidence = probs[pred_class]

print("Logits:", logits)
print("Probabilities:", probs)
print("Predicted class index:", pred_class)
print("Predicted label:", train_ds.class_names[pred_class])
print("Confidence:", confidence)